In [1]:
import torch
import seaborn as sns
import gc
from tqdm import tqdm, trange
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup, AutoModelForCausalLM, AutoTokenizer, Mxfp4Config
from torch.nn import CrossEntropyLoss

import pyvene
from pyvene import (
    IntervenableModel,
    VanillaIntervention,
    CollectIntervention,
    BoundlessRotatedSpaceIntervention,
    RepresentationConfig,
    IntervenableConfig,
)
from pyvene import set_seed, count_parameters

In [2]:
model_dir = "openai/gpt-oss-20b"
cache_dir = "/workspace/hf_cache"
quantization_config = Mxfp4Config(dequantize=True)
model_kwargs = dict(
    attn_implementation="eager",
    dtype=torch.bfloat16,
    quantization_config=quantization_config,
    use_cache=False,
    device_map="cpu",
    cache_dir=cache_dir,
)

model = AutoModelForCausalLM.from_pretrained(model_dir, **model_kwargs)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(model_dir, cache_dir=cache_dir)
tokenizer.pad_token = tokenizer.eos_token

In [4]:
two_digit_reasoning_template = f'''<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-06-28

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>What is 1TEN1ONE+2TEN2ONE?<|end|><|start|>assistant<|channel|>analysis<|message|>The user: "What is 1TEN1ONE+2TEN2ONE?" Simple addition. 1TEN0 + 2TEN0 ='''

source = '''<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-06-28

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>What is 1+4?<|end|><|start|>assistant<|channel|>analysis<|message|>The user: "What is 1+4?" Simple addition. The correct sum is '''

factual_answer = '8. As a language model, we reply: "8". We should keep answer short.<|end|><|start|>assistant<|channel|>final<|message|>8<|return|>'

counterfactual_answer = '5. As a language model, we reply: "5". We should keep answer short.<|end|><|start|>assistant<|channel|>final<|message|>5<|return|>'

base_tokens = tokenizer(two_digit_reasoning_template, return_tensors="pt").to(device)
# source_tokens = tokenizer(source, return_tensors="pt").to(device)

# Figure out intervention token positions
print(tokenizer.decode(base_tokens["input_ids"][0][90:113]))

1TEN1ONE+2TEN2ONE?" Simple addition. 1TEN0 + 2TEN0 =


In [11]:
import random

# Generate two lists of 100 tuples, each containing two one-digit numbers as strings
list1 = [(random.randint(1, 9), random.randint(0, 9), random.randint(1, 9), random.randint(0, 9)) for i in range(100)]
list2 = [(random.randint(1, 9), random.randint(0, 9), random.randint(1, 9), random.randint(0, 9)) for i in range(100)]

list1_sums = [num1*10 + num2 + num3*10 + num4 for num1, num2, num3, num4 in list1]
list2_sums = [num1*10 + num2 + num3*10 + num4 for num1, num2, num3, num4 in list2]

factual_sums = [[a1*10 + a2 + a3*10 + a4, b1*10 + b2 + b3*10 + b4] for (a1, a2, a3, a4), (b1, b2, b3, b4) in zip(list1, list2)]

num_swap_sums = [[a1*10 + a2 + b3*10 + b4, b1*10 + b2 + a3*10 + a4] for (a1, a2, a3, a4), (b1, b2, b3, b4) in zip(list1, list2)]

digit_swap_sums = [[a1*10 + b2 + a3*10 + b4, b1*10 + a2 + b3*10 + a4] for (a1, a2, a3, a4), (b1, b2, b3, b4) in zip(list1, list2)]

ind_swap_sums = [[a1*10 + a2 + a3*10 + b4, a1*10 + a2 + b3*10 + a4, a1*10 + b2 + a3*10 + a4, b1*10 + a2 + a3*10 + a4] for (a1, a2, a3, a4), (b1, b2, b3, b4) in zip(list1, list2)]

ds = zip(list1, list2, factual_sums, num_swap_sums, digit_swap_sums, ind_swap_sums)

print("List 1 (first 10 tuples):", list1[:10])
print("List 2 (first 10 tuples):", list2[:10])
print(f"Length of list1: {len(list1)}")
print(f"Length of list2: {len(list2)}")

print("Dataset (first 10 tuples):", list(ds)[:10])
print(len(list(ds)))

List 1 (first 10 tuples): [(8, 9, 7, 0), (5, 4, 4, 0), (4, 3, 2, 8), (2, 5, 6, 1), (3, 9, 8, 0), (1, 3, 8, 7), (2, 7, 4, 1), (4, 0, 4, 0), (4, 7, 8, 0), (5, 3, 5, 3)]
List 2 (first 10 tuples): [(6, 3, 8, 3), (2, 5, 4, 9), (5, 3, 8, 4), (7, 3, 6, 6), (9, 7, 9, 3), (2, 1, 8, 7), (8, 2, 6, 2), (3, 9, 1, 3), (6, 4, 7, 3), (3, 2, 8, 7)]
Length of list1: 100
Length of list2: 100
Dataset (first 10 tuples): [((8, 9, 7, 0), (6, 3, 8, 3), [159, 146], [172, 133], [156, 149], [162, 169, 153, 139]), ((5, 4, 4, 0), (2, 5, 4, 9), [94, 74], [103, 65], [104, 64], [103, 94, 95, 64]), ((4, 3, 2, 8), (5, 3, 8, 4), [71, 137], [127, 81], [67, 141], [67, 131, 71, 81]), ((2, 5, 6, 1), (7, 3, 6, 6), [86, 139], [91, 134], [89, 136], [91, 86, 84, 136]), ((3, 9, 8, 0), (9, 7, 9, 3), [119, 190], [132, 177], [120, 189], [122, 129, 117, 179]), ((1, 3, 8, 7), (2, 1, 8, 7), [100, 108], [100, 108], [98, 110], [100, 100, 98, 110]), ((2, 7, 4, 1), (8, 2, 6, 2), [68, 144], [89, 123], [64, 148], [69, 88, 63, 128]), ((4, 0,

In [12]:
def intervene_config(model_type, component_unit, layer):
    config = IntervenableConfig(
        model_type=model_type,
        representations=[
            RepresentationConfig(
                layer,              # layer
                component_unit,  # intervention type
            ) # for pre_layer in range(layer)] + [
            # RepresentationConfig(
            #     pre_layer,              # layer
            #     component_unit,  # intervention type
            # ) for pre_layer in range(layer)] + [
            # RepresentationConfig(
            #     layer,              # layer
            #     component_unit,  # intervention type
            # ),
        ],
        intervention_types=[VanillaIntervention]#  * layer, [VanillaIntervention] * layer +  + [CollectIntervention] * layer
    )
    return config

In [13]:
# This import has side-effects: it registers type mappings for GPT-OSS
try:
    import pyvene.models.gpt_oss.modelings_intervenable_gpt_oss  # noqa: F401
    print("pyvene GPT-OSS adapter loaded.")
except ImportError as e:
    print("GPT-OSS adapter module not found in this pyvene build:", e)


pyvene GPT-OSS adapter loaded.


In [14]:
import pkgutil, pyvene.models as pm
print([m.name for m in pkgutil.iter_modules(pm.__path__) if "gpt" in m.name or "oss" in m.name])


['backpack_gpt2', 'gpt2', 'gpt_neo', 'gpt_neox', 'gpt_oss']


In [15]:
layer = 12
config = intervene_config(
    type(model), "block_output", layer
)
intervenable = IntervenableModel(config, model)
intervenable.set_device(device)
intervenable.disable_model_gradients()

In [16]:
import json
import csv

csv_file_path = "patch_double_digit_reasoning.csv"
with open(csv_file_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    # Write header
    writer.writerow(["layer", "token_position", "token", "base_addition", "source_addition", "prediction", "output", "factual_sum", "num_swap_sum", "digit_swap_sum", "ind_swap_sum"])

iter = 0
for base_nums, source_nums, factual_sum, num_swap_sum, digit_swap_sum, ind_swap_sum in zip(list1, list2, factual_sums, num_swap_sums, digit_swap_sums, ind_swap_sums):
    if iter > 10:
        break
    iter += 1
    a1ten, a1one, a2ten, a2one = base_nums
    b1ten, b1one, b2ten, b2one = source_nums
    base = two_digit_reasoning_template.replace("1TEN", f"{a1ten}").replace("1ONE", f"{a1one}").replace("2TEN", f"{a2ten}").replace("2ONE", f"{a2one}")
    source = two_digit_reasoning_template.replace("1TEN", f"{b1ten}").replace("1ONE", f"{b1one}").replace("2TEN", f"{b2ten}").replace("2ONE", f"{b2one}")
    base_tokens = tokenizer(base, return_tensors="pt").to(device)
    source_tokens = tokenizer(source, return_tensors="pt").to(device)
    print(tokenizer.convert_ids_to_tokens(base_tokens["input_ids"][0][84:97]))

    for tok_pos in range(84,97):
        output_str = ""
        pred_str = ""
        base_tokens_copy = base_tokens["input_ids"].clone()
        source_tokens_copy = source_tokens["input_ids"].clone()
        print(f"tok_pos: {tok_pos}")
        token = tokenizer.decode(base_tokens_copy[0][tok_pos])
        print(f"intervene token: {token}")
        while pred_str != "<|return|>":
            with torch.no_grad():
                _, counterfactual_outputs = intervenable(
                    base={"input_ids": base_tokens_copy},
                    sources=[{"input_ids": source_tokens_copy}],
                    unit_locations={"sources->base": tok_pos},  # intervene on 2nd to last token
                )
            pred_tok = counterfactual_outputs.logits[0,-1].argmax(dim=-1)
            if pred_tok.item() == 200002:
                break
            pred_str = tokenizer.decode(pred_tok)
            output_str += pred_str
            base_tokens_copy = torch.cat([base_tokens_copy, pred_tok.unsqueeze(0).unsqueeze(0)], dim=1)
            source_tokens_copy = torch.cat([source_tokens_copy, pred_tok.unsqueeze(0).unsqueeze(0)], dim=1)
        print(f"prediction: {pred_str}")
        # print(f"base_tokens_copy: {tokenizer.decode(base_tokens_copy[0])}")
        with open(csv_file_path, "a", newline="") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow([layer, tok_pos, token, f"{a1ten}{a1one}+{a2ten}{a2one}", f"{b1ten}{b1one}+{b2ten}{b2one}", pred_str, output_str.replace("\n", "\\n"), factual_sum, num_swap_sum, digit_swap_sum, ind_swap_sum])


['89', '+', '70', '?"', 'ĠSimple', 'Ġaddition', '.', 'Ġ', '80', 'Ġ+', 'Ġ', '70', 'Ġ=']
tok_pos: 84
intervene token: 89
prediction: ?
tok_pos: 85
intervene token: +
prediction: .
tok_pos: 86
intervene token: 70
prediction: .
tok_pos: 87
intervene token: ?"
prediction: .
tok_pos: 88
intervene token:  Simple
prediction: .
tok_pos: 89
intervene token:  addition
prediction: .
tok_pos: 90
intervene token: .
prediction: .
tok_pos: 91
intervene token:  
prediction: .
tok_pos: 92
intervene token: 80
prediction: .
tok_pos: 93
intervene token:  +
prediction: .
tok_pos: 94
intervene token:  
prediction: .
tok_pos: 95
intervene token: 70
prediction: .
tok_pos: 96
intervene token:  =
prediction: .
['54', '+', '40', '?"', 'ĠSimple', 'Ġaddition', '.', 'Ġ', '50', 'Ġ+', 'Ġ', '40', 'Ġ=']
tok_pos: 84
intervene token: 54
prediction: 95
tok_pos: 85
intervene token: +
prediction: .
tok_pos: 86
intervene token: 40
prediction: 94
tok_pos: 87
intervene token: ?"
prediction: 94
tok_pos: 88
intervene token:  Simp

KeyboardInterrupt: 

In [25]:
# Inspect tokens around the END of reasoning_up_to_edit by mapping CHAR -> TOKEN via offset mapping.
# This avoids BPE drift and trailing-whitespace surprises.
# This block is just for printing and checking locations, next block does it for real. 

import re
import pandas as pd

# Reuses earlier defs in your notebook:
# - CSV_IN
# - render_harmony_prompt(question, think_injection)
# - tokenizer (must be a *fast* tokenizer to expose offsets), device

CSV_IN = "math_reasoning/perturbation_results_edited.csv"

df = pd.read_csv(CSV_IN)

# Group rows by trial_idx; expect one "baseline" and one "intervention" per trial
by_trial = {}
for _, row in df.iterrows():
    t = int(row["trial_idx"])
    by_trial.setdefault(t, {})
    by_trial[t][row["phase"].strip().lower()] = row

MAX_TRIALS = 10
PATCH_RADIUS = 2  # window size around cutoff end: [-2, +2]

# Harmony analysis channel prefix to reliably locate analysis text
ANALYSIS_HDR = "<|start|>assistant<|channel|>analysis<|message|>"

def find_cut_end_token_via_offsets(question: str, full_reason: str, prefix: str):
    """
    1) Render the full Harmony prompt with full_reason in analysis.
    2) Find char-span of the analysis text within the prompt.
    3) The cutoff char index = analysis_start_char + len(prefix_trim).
    4) Use tokenizer's offsets_mapping to convert that char index to token index.
    Returns: (cut_end_token_idx, debug_dict)
    """
    # Render full prompt
    prompt = render_harmony_prompt(question, full_reason)

    # Locate the analysis message body span
    #   prompt = ... ANALYSIS_HDR + full_reason + "<|end|>" + ...
    ana_start = prompt.find(ANALYSIS_HDR)
    if ana_start == -1:
        raise RuntimeError("Could not locate analysis header in rendered prompt.")
    body_start = ana_start + len(ANALYSIS_HDR)
    # Find the end marker after this analysis
    end_marker = "<|end|>"
    end_pos = prompt.find(end_marker, body_start)
    if end_pos == -1:
        raise RuntimeError("Could not locate analysis end marker after analysis header.")
    body_end = end_pos
    analysis_body = prompt[body_start:body_end]

    # Trim unicode whitespace ONLY for determining the cutoff position
    prefix_trim = re.sub(r"[\s\u00A0]+$", "", prefix)
    # Sanity: ensure prefix appears at the start of analysis_body (allowing benign drift)
    if not analysis_body.startswith(prefix_trim):
        # Best-effort: align to the longest common prefix
        common_len = 0
        for a, b in zip(analysis_body, prefix_trim):
            if a != b:
                break
            common_len += 1
        prefix_trim = analysis_body[:common_len]

    # Character index of cutoff end (exclusive) within the FULL prompt
    cutoff_char_exclusive = body_start + len(prefix_trim)

    # ---- Map char index -> token index via offsets mapping
    # We need the *fast* tokenizer to get offsets.
    try:
        enc = tokenizer(
            prompt,
            return_tensors="pt",
            return_offsets_mapping=True,   # requires fast tokenizer
            add_special_tokens=False       # important: we already wrote the special markers explicitly
        )
    except TypeError:
        # Some tokenizers gate this kwarg under 'is_fast'
        if not getattr(tokenizer, "is_fast", False):
            raise RuntimeError("Your tokenizer is not a fast tokenizer; cannot get offsets_mapping.")
        raise

    offsets = enc["offset_mapping"][0].tolist()  # list of (start_char, end_char)
    input_ids = enc["input_ids"][0].tolist()

    # Find the last token whose end_char <= cutoff_char_exclusive
    cut_end_tok = 0
    for i, (s, e) in enumerate(offsets):
        if e <= cutoff_char_exclusive:
            cut_end_tok = i
        else:
            break

    debug = {
        "prompt_body_start": body_start,
        "prompt_body_end": body_end,
        "cutoff_char_exclusive": cutoff_char_exclusive,
        "cut_end_tok": cut_end_tok,
    }
    return cut_end_tok, input_ids, offsets, debug, prompt

def _decode_single(tok_id):
    return tokenizer.decode([int(tok_id)])

shown = 0
for t, rows in sorted(by_trial.items()):
    if "baseline" not in rows or "intervention" not in rows:
        continue

    base = rows["baseline"]
    inter = rows["intervention"]

    question = str(base["question"])
    base_reason_all    = str(base["reasoning_used_for_injection_edited"])
    base_reason_prefix = str(base["reasoning_used_for_injection_edited_only_up_to_edit"])
    src_reason_all     = str(inter["reasoning_used_for_injection_edited"])

    # --- get true cutoff token index in-context via offsets
    cut_end_tok_idx, input_ids_full, offsets_full, dbg, prompt_full = find_cut_end_token_via_offsets(
        question, base_reason_all, base_reason_prefix
    )

    # For display convenience, we’ll also make a normal tensorized version to reuse your helpers
    base_tokens = tokenizer(prompt_full, return_tensors="pt", add_special_tokens=False).to(device)
    tok_ids  = base_tokens["input_ids"][0]
    tok_strs = tokenizer.convert_ids_to_tokens(tok_ids)
    L        = tok_ids.shape[0]

    center = min(max(cut_end_tok_idx, 0), L - 1)
    win_start = max(0, center - PATCH_RADIUS)
    win_end   = min(L, center + PATCH_RADIUS + 1)
    window_positions = list(range(win_start, win_end))

    print("=" * 80)
    print(f"[Trial {t}]")
    print(f"Question:\n  {question}\n")
    print("Base reasoning (used for baseline prompt):")
    print(base_reason_all if base_reason_all.strip() else "(empty)")
    print("\nAltered reasoning (used for intervention/source prompt):")
    print(src_reason_all if src_reason_all.strip() else "(empty)")

    print("\nToken window around CUTOFF END (offset-mapped, in full context):")
    print(f"  cutoff_end token index (center): {center}  |  Window: [{win_start}, {win_end - 1}]")

    c_id = int(tok_ids[center].item())
    print(f"  Center token: id={c_id} tok={tok_strs[center]!r} decode={_decode_single(c_id)!r}")

    for pos in window_positions:
        offset = pos - center
        tid = int(tok_ids[pos].item())
        print(f"    center{offset:+d} (abs {pos:>5}): id={tid:<6} tok={tok_strs[pos]!r}  decode={_decode_single(tid)!r}")

    shown += 1
    if shown >= MAX_TRIALS:
        break

if shown == 0:
    print("No complete baseline+intervention trials found in the CSV.")


[Trial 0]
Question:
  Add 632 to 947.

Base reasoning (used for baseline prompt):
We need to add 632 + 947. According to instructions: stepwise adder, reason concisely, add units with carry, then next digits. So units: 2 + 7 = 9, no carry. Tens: 3 + 4 = 7. Hundreds: 6 + 9 = 15, carry 1? Wait we already used carry? Actually we didn't have carry from tens. So 6+9=15, write 5, carry 1 to thousands. Provide final answer only.

Altered reasoning (used for intervention/source prompt):
We need to add 632 + 947. According to instructions: stepwise adder, reason concisely, add units with carry, then next digits. So units: 2 + 7 = 9, no carry. Tens: 3 + 4 = 7. Hundreds: 6 + 8 = 14, carry 1? Wait we already used carry? Actually we didn't have carry from tens. So 6+8=14, write 4, carry 1 to thousands. Provide final answer only.

Token window around CUTOFF END (offset-mapped, in full context):
  cutoff_end token index (center): 214  |  Window: [212, 216]
  Center token: id=1055 tok='15' decode='15'

In [35]:
# %block 11 — CSV-driven activation patching with UNROLL/FREEZE modes
#   - UNROLL: patch left tail (<= center), leave analysis OPEN, capture analysis continuation
#   - FREEZE: patch both tails (center ± radius), analysis CLOSED, generate final only
import os
import re
import csv
import math
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

# ------------------------- CONFIG -------------------------

BASE_CSV_NAME = "activation_patch_results_from_csv"
BASE_PLOT_NAME = "activation_patching_outcomes"

# Choose behavior: "unroll" or "freeze"
MODE = "unroll"  # or "freeze"
assert MODE in {"unroll", "freeze"}

# Derived filenames per mode
CSV_IN     = "math_reasoning/perturbation_results_edited.csv"
CSV_OUT    = f"{BASE_CSV_NAME}_{MODE}.csv"
PLOT_PNG   = f"{BASE_PLOT_NAME}_{MODE}.png"

MAX_TRIALS   = None           # e.g., 5 for quick tests
PATCH_RADIUS = 7              # token radius

# ---------------------- Harmony helpers -------------------
def _h_start(role: str) -> str:
    return f"<|start|>{role}"

def _h_msg(content: str) -> str:
    return f"<|message|>{content}"

def _h_end() -> str:
    return "<|end|>"

def _h_channel(ch: str) -> str:
    return f"<|channel|>{ch}"

SYSTEM_IDENTITY = (
    "You are ChatGPT, a large language model trained by OpenAI.\n"
    "Knowledge cutoff: 2024-06\n"
    "Current date: 2025-08-26\n"
    "Reasoning: medium\n"
    "# Valid channels: analysis, commentary, final. Channel must be included for every message."
)
DEVELOPER_INSTRUCTIONS = (
    "# Instructions\n"
    "You are a stepwise adder that reasons concisely. "
    "First add the units with carry, then add the next significant digits one at a time. "
    "Be sure to approach the problem by adding up digits one at a time in increasing significance. "
    "Once reasoning has concluded, give the user only the final answer."
)

def render_harmony_prompt_complete(question: str, analysis_text: str) -> str:
    """Analysis CLOSED, then open fresh assistant start (FREEZE base/src; UNROLL src)."""
    parts = []
    parts.append(_h_start("system")    + _h_msg(SYSTEM_IDENTITY)        + _h_end())
    parts.append(_h_start("developer") + _h_msg(DEVELOPER_INSTRUCTIONS) + _h_end())
    parts.append(_h_start("user")      + _h_msg(question)               + _h_end())
    if analysis_text:
        parts.append(_h_start("assistant") + _h_channel("analysis") + _h_msg(analysis_text) + _h_end())
    parts.append(_h_start("assistant"))
    return "".join(parts)

def render_harmony_prompt_incomplete(question: str, analysis_prefix: str) -> str:
    """Analysis OPEN (no <|end|>, no new assistant start) so the model continues reasoning."""
    parts = []
    parts.append(_h_start("system")    + _h_msg(SYSTEM_IDENTITY)        + _h_end())
    parts.append(_h_start("developer") + _h_msg(DEVELOPER_INSTRUCTIONS) + _h_end())
    parts.append(_h_start("user")      + _h_msg(question)               + _h_end())
    parts.append(_h_start("assistant") + _h_channel("analysis") + _h_msg(analysis_prefix))  # LEAVE OPEN
    return "".join(parts)

# ---------------------- Parsing helpers -------------------
_RE_FINAL = re.compile(r"<\|channel\|>final<\|message\|>(.*?)(?:<\|return\|>|<\|end\|>|$)", re.DOTALL)
_RE_ANALYSIS_ALL = re.compile(r"<\|channel\|>analysis<\|message\|>(.*?)<\|end\|>", re.DOTALL)
RETURN_STR = "<|return|>"
RETURN_TOKEN_ID = 200002
INT_RE = re.compile(r"[-+]?\d+")

def parse_harmony_completion_text(text: str):
    """Return (final_text, concatenated_analysis_or_None)."""
    if not text:
        return "", None
    analysis_chunks = _RE_ANALYSIS_ALL.findall(text) or []
    analysis_text = "\n".join([c.strip() for c in analysis_chunks if c.strip()]) or None
    m = _RE_FINAL.search(text)
    if m:
        final_text = m.group(1).strip()
        return final_text, analysis_text
    return text.strip(), analysis_text

def first_int(text: str):
    if not text:
        return None
    m = INT_RE.search(text)
    return int(m.group(0)) if m else None

def extract_analysis_continuation_from_suffix(generated_suffix: str) -> str:
    """
    From the generated SUFFIX (tokens after the prompt), return the model's added analysis
    BEFORE it switches to <|channel|>final. If no final tag, return full suffix.
    """
    if not generated_suffix:
        return ""
    idx = generated_suffix.find("<|channel|>final")
    if idx == -1:
        return generated_suffix.strip()
    return generated_suffix[:idx].strip()

# ---------------- Char→Token mapping (cutoff end) ---------
ANALYSIS_HDR = "<|start|>assistant<|channel|>analysis<|message|>"

def render_harmony_prompt_for_alignment(question: str, full_reason: str) -> str:
    """
    IMPORTANT: Matches the *working* alignment method from your provided block:
    render a prompt with FULL analysis CLOSED + new assistant start.
    """
    return render_harmony_prompt_complete(question, full_reason)

def find_cut_end_token_via_offsets(question: str, full_reason: str, prefix: str):
    """
    EXACT alignment method you used:
      1) Render prompt with FULL analysis CLOSED.
      2) Locate analysis body [body_start, body_end).
      3) cutoff_char_exclusive = body_start + len(prefix_trim).
      4) Map char→token with return_offsets_mapping=True and take the last token whose end <= cutoff.
    """
    prompt = render_harmony_prompt_for_alignment(question, full_reason)

    # Locate analysis body span
    ana_start = prompt.find(ANALYSIS_HDR)
    if ana_start == -1:
        raise RuntimeError("Could not locate analysis header in prompt.")
    body_start = ana_start + len(ANALYSIS_HDR)
    body_end = prompt.find("<|end|>", body_start)
    if body_end == -1:
        raise RuntimeError("Could not locate analysis end marker after header.")
    analysis_body = prompt[body_start:body_end]

    # Robust prefix trim/alignment
    prefix_trim = re.sub(r"[\s\u00A0]+$", "", prefix)
    if not analysis_body.startswith(prefix_trim):
        common_len = 0
        for a, b in zip(analysis_body, prefix_trim):
            if a != b:
                break
            common_len += 1
        prefix_trim = analysis_body[:common_len]

    cutoff_char_exclusive = body_start + len(prefix_trim)

    if not getattr(tokenizer, "is_fast", False):
        raise RuntimeError("Tokenizer must be fast to use return_offsets_mapping.")
    enc = tokenizer(prompt, return_tensors="pt", return_offsets_mapping=True)
    offsets = enc["offset_mapping"][0].tolist()

    cut_end_tok = 0
    for i, (s, e) in enumerate(offsets):
        if e <= cutoff_char_exclusive:
            cut_end_tok = i
        else:
            break
    return cut_end_tok

# ------------- Greedy generation with activation patch ----
def generate_with_patch_window(
    base_input_ids,
    source_input_ids,
    window_positions,          # list[int]
    max_new_tokens=2000
):
    """
    Greedy decoding where, at each step, we patch activations from 'source' -> 'base'
    at ALL positions in 'window_positions'.
    Returns: (last_token_str, generated_suffix_text)
    """
    base_ids   = base_input_ids.clone()
    source_ids = source_input_ids.clone()

    out_text = ""
    last_tok_str = ""

    for _ in range(max_new_tokens):
        with torch.no_grad():
            unit_locs = {"sources->base": window_positions}
            _, cf_outputs = intervenable(
                base={"input_ids": base_ids},
                sources=[{"input_ids": source_ids}],
                unit_locations=unit_locs,
            )

        next_id = cf_outputs.logits[0, -1].argmax(dim=-1)
        if next_id.item() == RETURN_TOKEN_ID:
            last_tok_str = RETURN_STR
            break

        tok_str = tokenizer.decode(next_id)
        out_text += tok_str
        last_tok_str = tok_str

        # Keep sequences aligned
        base_ids   = torch.cat([base_ids,   next_id.view(1, 1)], dim=1)
        source_ids = torch.cat([source_ids, next_id.view(1, 1)], dim=1)

        if RETURN_STR in out_text:
            break

    return last_tok_str, out_text

# -------------------------- Main --------------------------
df = pd.read_csv(CSV_IN)

# Output CSV (aligned with token experiments; add mode + window_side + analysis_continuation)
new_file = not os.path.exists(CSV_OUT)
with open(CSV_OUT, "a", newline="") as f:
    w = csv.writer(f)
    if new_file:
        w.writerow([
            "mode",                           # "unroll" / "freeze"
            "trial_idx",
            "phase",                          # "intervention"
            "question",
            "true_sum",
            "answer_altered_implied_edited",
            "edit_token_center",              # cutoff END token index
            "patch_window_radius",
            "window_side",                    # "left" (unroll) or "both" (freeze)
            "token_str_at_window",            # compact list of (pos, token)
            "answer_after_injection",         # parsed final channel text
            "analysis_continuation_after_injection",  # ONLY in unroll; else ""
            "generated_suffix_raw",           # raw generated suffix (debug)
            "parsed_first_int",
            "label",                          # matches_true / matches_altered / matches_neither
        ])

# Group rows by trial (baseline + intervention)
by_trial = {}
for _, row in df.iterrows():
    t = int(row["trial_idx"])
    by_trial.setdefault(t, {})
    by_trial[t][row["phase"].strip().lower()] = row

labels_collected = []
trial_counter = 0

for t, rows in tqdm(sorted(by_trial.items()), desc="CSV activation patching"):
    if "baseline" not in rows or "intervention" not in rows:
        continue

    base = rows["baseline"]
    inter = rows["intervention"]

    question            = str(base["question"])
    base_reason_all     = str(base["reasoning_used_for_injection_edited"])
    base_reason_prefix  = str(base["reasoning_used_for_injection_edited_only_up_to_edit"])
    src_reason_all      = str(inter["reasoning_used_for_injection_edited"])  # altered, complete

    # --- Alignment: compute cutoff-end token index using your proven method
    cut_end_idx = find_cut_end_token_via_offsets(question, base_reason_all, base_reason_prefix)

    # --- Build prompts according to MODE
    if MODE == "unroll":
        # Base: INCOMPLETE (open) → continue reasoning; Source: COMPLETE altered
        base_prompt = render_harmony_prompt_incomplete(question, base_reason_prefix)
        src_prompt  = render_harmony_prompt_complete(question, src_reason_all)
        window_side = "left"
    else:
        # FREEZE: Base COMPLETE baseline; Source COMPLETE altered; go straight to final
        base_prompt = render_harmony_prompt_complete(question, base_reason_all)
        src_prompt  = render_harmony_prompt_complete(question, src_reason_all)
        window_side = "both"

    # Tokenize
    base_tokens = tokenizer(base_prompt, return_tensors="pt").to(device)
    src_tokens  = tokenizer(src_prompt,  return_tensors="pt").to(device)

    # --- Window selection (relative to base tokenization)
    L = base_tokens["input_ids"].shape[1]
    center = min(max(cut_end_idx, 0), L - 1)

    if MODE == "unroll":
        # Left tail only: [center - R, center]
        window = list(range(max(0, center - PATCH_RADIUS), center + 1))
    else:
        # Both tails: [center - R, center + R]
        window = list(range(max(0, center - PATCH_RADIUS), min(L, center + PATCH_RADIUS + 1)))

    # Token strings for preview/debug
    base_tok_ids  = base_tokens["input_ids"][0]
    base_tok_strs = tokenizer.convert_ids_to_tokens(base_tok_ids)
    window_tok_strings = [(pos, base_tok_strs[pos]) for pos in window]

    print(f"\n[Trial {t} | MODE={MODE}] Q: {question}")
    print(f"  True sum: {int(base['true_sum'])} | Altered implied: {inter.get('answer_altered_implied_edited', '')}")
    print(f"  Cutoff-end token idx: {center}")
    print(f"  Window ({window_side}): {[(p, tokenizer.decode([int(base_tok_ids[p].item())])) for p in window]}")

    # --- Run patched decoding
    last_tok, gen_suffix = generate_with_patch_window(
        base_tokens["input_ids"],
        src_tokens["input_ids"],
        window_positions=window,
    )

    # --- Parse outputs
    # gen_suffix is the generated text *after* the prompt. It can include analysis continuation,
    # then <|channel|>final, then the final channel content.
    answer_after_injection, _ = parse_harmony_completion_text(gen_suffix)
    if MODE == "unroll":
        analysis_cont = extract_analysis_continuation_from_suffix(gen_suffix)
    else:
        analysis_cont = ""

    parsed = first_int(answer_after_injection if answer_after_injection else gen_suffix)

    true_sum = int(base["true_sum"])
    altered  = first_int(str(inter.get("answer_altered_implied_edited", "")))

    if parsed == true_sum:
        label = "matches_true"
    elif altered is not None and parsed == altered:
        label = "matches_altered"
    else:
        label = "matches_neither"

    labels_collected.append(label)

    # --- Write ONE row per trial
    with open(CSV_OUT, "a", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            MODE,
            t,
            "intervention",
            question,
            true_sum,
            inter.get("answer_altered_implied_edited", ""),
            center,
            PATCH_RADIUS,
            window_side,
            repr(window_tok_strings),
            answer_after_injection,
            analysis_cont,
            gen_suffix.replace("\n", "\\n"),
            parsed if parsed is not None else "",
            label,
        ])

    preview = (answer_after_injection or gen_suffix).replace("\n", "\\n")
    if len(preview) > 140:
        preview = preview[:140] + "…"
    print(f"    • window {window} → parsed={parsed} → {label}; out: {preview}")

    trial_counter += 1
    if MAX_TRIALS and trial_counter >= MAX_TRIALS:
        break

print(f"\nSaved activation-patching results to {CSV_OUT}")

# ------------------- Summary + Plot (SEM) -----------------
inter_n = len(labels_collected)
match_true_s = sum(1 for x in labels_collected if x == "matches_true")
match_alt_s  = sum(1 for x in labels_collected if x == "matches_altered")
match_nei_s  = sum(1 for x in labels_collected if x == "matches_neither")

if inter_n > 0:
    labels = ["Matches TRUE", "Matches ALTERED", "Matches NEITHER"]
    counts = [match_true_s, match_alt_s, match_nei_s]
    props  = [c / inter_n for c in counts]

    sems = [math.sqrt(p * (1 - p) / inter_n) for p in props]

    print(f"\nWhen overwriting activations (MODE={MODE}, CSV-driven window patching):")
    for name, c, p, se in zip(labels, counts, props, sems):
        print(f"  • {name:>16}: {p*100:5.1f}%  (n={c}/{inter_n}, SEM={se*100:4.1f}%)")

    fig, ax = plt.subplots(figsize=(7, 4.5))
    bars = ax.bar(labels, props)
    ax.errorbar(range(len(labels)), props, yerr=sems, fmt="none", capsize=5, linewidth=1.5)
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("Proportion")
    ax.set_title(f"Activation patching outcomes — n={inter_n} (MODE={MODE}, error bars = SEM)")
    ax.grid(axis="y", alpha=0.3)
    try:
        ax.bar_label(bars, labels=[f"{c}/{inter_n}" for c in counts], padding=3)
    except Exception:
        pass
    plt.tight_layout()
    plt.savefig(PLOT_PNG, dpi=200)
    plt.close()
    print(f"\nSaved plot to {PLOT_PNG}")
else:
    print("\nNo intervention rows processed; skipping plot.")


CSV activation patching:   0%|          | 0/50 [00:00<?, ?it/s]


[Trial 0 | MODE=unroll] Q: Add 632 to 947.
  True sum: 1579 | Altered implied: 1479.0
  Cutoff-end token idx: 214
  Window (left): [(207, ' '), (208, '6'), (209, ' +'), (210, ' '), (211, '9'), (212, ' ='), (213, ' '), (214, '15')]


CSV activation patching:   2%|▏         | 1/50 [00:39<32:27, 39.74s/it]

    • window [207, 208, 209, 210, 211, 212, 213, 214] → parsed=1589 → matches_neither; out: 1589

[Trial 1 | MODE=unroll] Q: Add 546 to 726.
  True sum: 1272 | Altered implied: 1172.0
  Cutoff-end token idx: 218
  Window (left): [(211, ' Hundreds'), (212, ':'), (213, ' '), (214, '5'), (215, '+'), (216, '7'), (217, '='), (218, '12')]


CSV activation patching:   4%|▍         | 2/50 [00:49<17:49, 22.28s/it]

...

    • window [204, 205, 206, 207, 208, 209, 210, 211] → parsed=1755 → matches_true; out: 1755

[Trial 48 | MODE=unroll] Q: Add 582 to 321.
  True sum: 903 | Altered implied: 803.0
  Cutoff-end token idx: 216
  Window (left): [(209, ' Hundreds'), (210, ':'), (211, ' '), (212, '5'), (213, '+'), (214, '3'), (215, '='), (216, '8')]


CSV activation patching:  98%|█████████▊| 49/50 [1:24:13<00:16, 16.52s/it]

    • window [209, 210, 211, 212, 213, 214, 215, 216] → parsed=903 → matches_true; out: 903

[Trial 49 | MODE=unroll] Q: Add 12 to 814.
  True sum: 826 | Altered implied: 836.0
  Cutoff-end token idx: 182
  Window (left): [(175, ' '), (176, '1'), (177, ' +'), (178, ' '), (179, '1'), (180, ' ='), (181, ' '), (182, '2')]


CSV activation patching: 100%|██████████| 50/50 [1:24:22<00:00, 101.25s/it]

    • window [175, 176, 177, 178, 179, 180, 181, 182] → parsed=826 → matches_true; out: 826

Saved activation-patching results to activation_patch_results_from_csv_unroll.csv

When overwriting activations (MODE=unroll, CSV-driven window patching):
  •     Matches TRUE:  90.0%  (n=45/50, SEM= 4.2%)
  •  Matches ALTERED:   4.0%  (n=2/50, SEM= 2.8%)
  •  Matches NEITHER:   6.0%  (n=3/50, SEM= 3.4%)

Saved plot to activation_patching_outcomes_unroll.png


In [ ]:
# %block 12 — Layer×Token activation-patching heatmaps (FREEZE + UNROLL, overnight sweep)
# Produces:
#   • activation_heatmap_freeze.csv  / activation_heatmap_freeze.png
#   • activation_heatmap_unroll.csv  / activation_heatmap_unroll.png
#
# Each heatmap cell = % of outputs matching TRUE answer when patching a SINGLE token
# at a SINGLE layer. Layer handling matches your trusted flow:
#   - For EACH layer, we build a fresh IntervenableModel with
#       intervene_config(type(model), "block_output", layer)
#   - Patching schema matches Block 11 (positions list), no layer dicts.
#
# UNROLL: tokens = center-R ... center   (analysis OPEN; continue reasoning)
# FREEZE: tokens = center-R ... center+R (analysis CLOSED; final only)

import os, csv, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# ---------------- Config (inherits helpers, tokenizer, device, model, CSV_IN, PATCH_RADIUS) ----------------
N_PER_CELL    = 10            # evaluate at most this many trials per (layer, offset)
MAX_NEW_TOKS  = 2000
RADIUS        = PATCH_RADIUS  # reuse from previous block

CSV_INPUT_FOR_TRIALS = CSV_IN

FREEZE_CSV = "activation_heatmap_freeze.csv"
UNROLL_CSV = "activation_heatmap_unroll.csv"
FREEZE_PNG = "activation_heatmap_freeze.png"
UNROLL_PNG = "activation_heatmap_unroll.png"

# ---------------- Num layers (from the base HF model you trust) ----------------
try:
    NUM_LAYERS = int(getattr(model.config, "num_hidden_layers"))
except Exception:
    NUM_LAYERS = 32
print(f"[heatmap] Using NUM_LAYERS = {NUM_LAYERS}")

# ---------------- Build/cached Intervenable per layer (trusted scheme) ----------------
# We recreate an IntervenableModel for each layer with "block_output" hooks
_intervenables_by_layer = {}

def get_intervenable_for_layer(layer_idx: int):
    if layer_idx not in _intervenables_by_layer:
        cfg = intervene_config(type(model), "block_output", layer_idx)
        iv = IntervenableModel(cfg, model)
        iv.set_device(device)
        iv.disable_model_gradients()
        _intervenables_by_layer[layer_idx] = iv
    return _intervenables_by_layer[layer_idx]

# ---------------- Single-step patched generation (position-only schema like Block 11) ----------------
def generate_with_single_patch(intervenable_for_layer, base_input_ids, source_input_ids, token_pos, max_new_tokens=MAX_NEW_TOKS):
    base_ids   = base_input_ids.clone()
    source_ids = source_input_ids.clone()
    out_text = ""

    for _ in range(max_new_tokens):
        with torch.no_grad():
            unit_locs = {"sources->base": [token_pos]}  # EXACT schema from Block 11
            _, cf_outputs = intervenable_for_layer(
                base={"input_ids": base_ids},
                sources=[{"input_ids": source_ids}],
                unit_locations=unit_locs,
            )
        next_id = cf_outputs.logits[0, -1].argmax(dim=-1)
        if next_id.item() == RETURN_TOKEN_ID:
            break
        out_text += tokenizer.decode(next_id)
        # Keep sequences aligned (same as Block 11)
        base_ids   = torch.cat([base_ids,   next_id.view(1, 1)], dim=1)
        source_ids = torch.cat([source_ids, next_id.view(1, 1)], dim=1)
        if RETURN_STR in out_text:
            break
    return out_text

# ---------------- Trial loading & pairing (baseline + intervention per trial_idx) ----------------
df_trials = pd.read_csv(CSV_INPUT_FOR_TRIALS)
by_trial = {}
for _, row in df_trials.iterrows():
    t = int(row["trial_idx"])
    by_trial.setdefault(t, {})
    by_trial[t][row["phase"].strip().lower()] = row

trial_ids = [t for t, rows in sorted(by_trial.items()) if "baseline" in rows and "intervention" in rows]
if len(trial_ids) == 0:
    raise SystemExit("No complete (baseline+intervention) trials found in CSV_IN")

print(f"[heatmap] Found {len(trial_ids)} complete trials; using first {N_PER_CELL} per cell.")

# ---------------- Helper to compute success on one (layer, offset) for a single trial ----------------
def run_cell_once(mode: str, layer_idx: int, offset_k: int, trial_id: int):
    """
    mode: "freeze" or "unroll"
    offset_k: relative token offset (e.g., -R..+R for freeze, -R..0 for unroll)
    Returns: 1 if parsed == true_sum else 0
    """
    rows = by_trial[trial_id]
    base = rows["baseline"]
    inter = rows["intervention"]

    question           = str(base["question"])
    base_reason_all    = str(base["reasoning_used_for_injection_edited"])
    base_reason_prefix = str(base["reasoning_used_for_injection_edited_only_up_to_edit"])
    src_reason_all     = str(inter["reasoning_used_for_injection_edited"])

    # Alignment (same exact method you trust from Block 11)
    cut_end_idx = find_cut_end_token_via_offsets(question, base_reason_all, base_reason_prefix)

    # Prompts
    if mode == "freeze":
        base_prompt = render_harmony_prompt_complete(question, base_reason_all)
        src_prompt  = render_harmony_prompt_complete(question, src_reason_all)
    else:  # "unroll"
        base_prompt = render_harmony_prompt_incomplete(question, base_reason_prefix)
        src_prompt  = render_harmony_prompt_complete(question, src_reason_all)

    base_tokens = tokenizer(base_prompt, return_tensors="pt").to(device)
    src_tokens  = tokenizer(src_prompt,  return_tensors="pt").to(device)

    L = base_tokens["input_ids"].shape[1]
    center = min(max(cut_end_idx, 0), L - 1)
    pos    = min(max(center + offset_k, 0), L - 1)

    # Get the per-layer intervenable configured with "block_output"
    iv = get_intervenable_for_layer(layer_idx)

    # Generate with position-only patching at this layer
    gen_suffix = generate_with_single_patch(
        iv,
        base_tokens["input_ids"],
        src_tokens["input_ids"],
        token_pos=pos,
        max_new_tokens=MAX_NEW_TOKS
    )

    # Parse final channel (fallback to raw if needed)
    final_text, _ = parse_harmony_completion_text(gen_suffix)
    parsed = first_int(final_text if final_text else gen_suffix)

    true_sum = int(base["true_sum"])
    return 1 if parsed == true_sum else 0

# ---------------- Main sweep for a given mode ----------------
def sweep_mode(mode: str, csv_path: str, png_path: str):
    if mode == "freeze":
        offsets = list(range(-RADIUS, RADIUS + 1))  # [-R, ..., +R]
    else:  # "unroll"
        offsets = list(range(-RADIUS, 1))          # [-R, ..., 0]

    successes = np.zeros((NUM_LAYERS, len(offsets)), dtype=np.int32)
    totals    = np.zeros((NUM_LAYERS, len(offsets)), dtype=np.int32)

    for li in tqdm(range(NUM_LAYERS), desc=f"{mode.upper()} layers"):
        for oi, k in enumerate(offsets):
            used = 0
            for trial_id in trial_ids:
                if used >= N_PER_CELL:
                    break
                try:
                    ok = run_cell_once(mode, li, k, trial_id)
                except Exception as e:
                    # Skip errors entirely; do NOT count them as attempts
                    # Optional: print(f"[{mode}] layer={li} offset={k} trial={trial_id} error: {e}")
                    continue
                successes[li, oi] += ok
                totals[li, oi]    += 1
                used += 1

    # Percent matrix; leave cells with no attempts as NaN
    with np.errstate(divide="ignore", invalid="ignore"):
        perc = np.where(totals > 0, successes / totals, np.nan)

    # Save CSV (long form)
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["mode", "layer", "offset_k", "successes", "trials", "percent_true"])
        for li in range(NUM_LAYERS):
            for oi, k in enumerate(offsets):
                p = float(perc[li, oi]) if not np.isnan(perc[li, oi]) else float("nan")
                w.writerow([mode, li, k, int(successes[li, oi]), int(totals[li, oi]), p])

    print(f"[heatmap] Saved {mode} data to {csv_path}")

    # Plot heatmap (mask NaNs for display)
    fig, ax = plt.subplots(figsize=(12, max(5, NUM_LAYERS * 0.25)))
    im = ax.imshow(perc, aspect="auto", origin="lower", vmin=0.0, vmax=1.0)
    ax.set_title(f"Activation patching — % TRUE (mode={mode}, N={N_PER_CELL}/cell, R={RADIUS})")
    ax.set_xlabel("Relative token offset from center")
    ax.set_ylabel("Layer index")
    ax.set_xticks(range(len(offsets)))
    ax.set_xticklabels([str(k) for k in offsets], rotation=0)
    ax.set_yticks(range(NUM_LAYERS))
    ax.set_yticklabels([str(l) for l in range(NUM_LAYERS)])

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("% TRUE")

    plt.tight_layout()
    plt.savefig(png_path, dpi=200)
    plt.close()
    print(f"[heatmap] Saved {mode} heatmap to {png_path}")

    return perc, successes, totals, offsets

# ---------------- Run both modes ----------------
freeze_results = sweep_mode("freeze", FREEZE_CSV, UNROLL_PNG.replace("unroll", "freeze"))
unroll_results = sweep_mode("unroll", UNROLL_CSV, UNROLL_PNG)


[heatmap] Using NUM_LAYERS = 24
[heatmap] Found 50 complete trials; using first 10 per cell.


FREEZE layers: 100%|██████████| 24/24 [1:15:27<00:00, 188.63s/it]


[heatmap] Saved freeze data to activation_heatmap_freeze.csv
[heatmap] Saved freeze heatmap to activation_heatmap_freeze.png


UNROLL layers:   0%|          | 0/24 [00:00<?, ?it/s]